# MAPED on CPU

Load five synthetic acquisitions, align detector and scan shifts, merge, and inspect the result. Run `python make_data.py synthetic-tilts` once first. See [README](README.md) for the required development environment. No experimental data is used.

The input adapts the [PR #256 validation example](https://github.com/electronmicroscopy/quantem/pull/256): five 64 × 64 scans, each containing 64 × 64 diffraction patterns. All coordinates are pixels.

In [ ]:
from pathlib import Path

from quantem.core.datastructures import Dataset4dstem
from quantem.core.io import load
from quantem.diffraction import MAPED

folder = Path("synthetic-tilts")

## Load

Use native QuantEM on CPU. Neither quantem.gpu nor quantem.widget is required. The synthetic inputs need no detector-mask correction.

Loading uses QuantEM’s existing RosettaSciIO reader. A warning about a missing microscope metadata companion means physical calibration was not supplied; these synthetic scans use pixel coordinates.

In [ ]:
files = sorted(folder.glob("*_master.h5"))
tilts = [Dataset4dstem.from_file(file, "arina") for file in files]
maped = MAPED.from_datasets(tilts)
maped

## Align

Find each direct beam, align the diffraction patterns, then align the scan images. The table reports the applied **(row, col)** corrections in pixels; a common translation is arbitrary.

In [ ]:
maped.find_beam_center(show=False)
maped.align_diffraction(show=False)

In [ ]:
maped.align_real_space(iterations=80, show=False)
maped.report()

## Inspect individual diffraction patterns

Compare one input tilt with the merged result at three marked specimen positions, using the same scan and detector alignment. Patterns come from individual positions, with a shared colour range. Coordinates are (row, col) in the cropped merged scan.

In [ ]:
merged = maped.merge(crop=True, show=False)
maped.show(positions=[(15, 15), (25, 25), (40, 40)], tilt_indices=[0])

## Save and reopen

Native QuantEM serialization retains the float32 values and alignment metadata in a Zarr ZIP. The input files remain unchanged.

In [ ]:
output = Path("merged_cpu.zip")
merged.save(output)
reopened = load(output)
reopened

For the same workflow in one call: `MAPED.from_datasets(tilts).run(iterations=80, show=False)`. Exact float32 output is limited to 4 GiB; see the README before using a full experimental scan.